# SAM + augmentation consistency pipeline on Colab

Runs the full pipeline (no SAM3D) on 100–150 images:
1. Clone repo (or use your GitHub URL)
2. Install deps + download SAM checkpoint
3. Download MIT Indoor, sample 100–150 images (3 classes)
4. Run SAM on originals → masks + metadata
5. Generate augmented images (even: hflip / rotation / resize / blur)
6. Run SAM on augmented images
7. Evaluate consistency → analysis → plots

**Runtime:** GPU recommended (faster SAM). Results can be zipped and downloaded.

## 1. Clone repo and go to project dir

Push your `sam-household-masks` code to a GitHub repo, then paste the clone URL below (use HTTPS, e.g. `https://github.com/USERNAME/sam-household-masks.git`). If you already have the repo elsewhere, you can skip clone and set `PROJECT_DIR` manually.

In [1]:
REPO_URL = "https://github.com/manga-sonta/SAM_Household_Masks.git"

import os

!git clone --depth 1 $REPO_URL

PROJECT_DIR = "/content/SAM_Household_Masks"

os.chdir(PROJECT_DIR)
print("Project dir:", os.getcwd())

Cloning into 'SAM_Household_Masks'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 156 (delta 1), reused 153 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 19.30 MiB | 40.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Project dir: /content/SAM_Household_Masks


In [2]:
!ls

analyze_consistency.py	  download_mit_indoor.py   plot_consistency.py
batch_sam_masks.py	  evaluate_consistency.py  README.md
colab_run_pipeline.ipynb  generate_augmented.py    requirements.txt
doc			  images		   run_sam3d.py


## 2. Install dependencies and SAM checkpoint

In [ ]:
!pip install -q torch torchvision numpy opencv-python Pillow tqdm scipy matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

  Preparing metadata (setup.py) ... done


In [ ]:
# Download SAM checkpoint (ViT-B, ~375 MB)
CHECKPOINT = "sam_vit_b_01ec64.pth"
if not os.path.isfile(os.path.join(PROJECT_DIR, CHECKPOINT)):
    !wget -q -O {CHECKPOINT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Downloaded", CHECKPOINT)
else:
    print("Checkpoint already present:", CHECKPOINT)

Downloaded sam_vit_b_01ec64.pth


## 3. Download MIT Indoor and sample 100–150 images (3 classes)

In [3]:
import os
import glob

IMAGE_DIR = os.path.join(PROJECT_DIR, "images", "mit_indoor_subset")

images = glob.glob(os.path.join(IMAGE_DIR, "*.*"))
print("Images found:", len(images))

print("Example images:")
print(images[:5])

Images found: 120
Example images:
['/content/SAM_Household_Masks/images/mit_indoor_subset/livingroom__int588.jpg', '/content/SAM_Household_Masks/images/mit_indoor_subset/bedroom__bed213.jpg', '/content/SAM_Household_Masks/images/mit_indoor_subset/livingroom__b_Ruth_House_Living_Room.jpg', '/content/SAM_Household_Masks/images/mit_indoor_subset/livingroom__cdMC1284.jpg', '/content/SAM_Household_Masks/images/mit_indoor_subset/bedroom__b23.jpg']


## 4. Run SAM on original images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_subset --output sam_output/original --checkpoint /content/sam_vit_b_01ec64.pth --model vit_b

Images: 100% 120/120 [50:05<00:00, 25.05s/it]
Done. Processed 120 images. Output: sam_output/original


## 5. Generate augmented images (even: hflip, rotation, resize, blur)

In [ ]:
!python generate_augmented.py --images images/mit_indoor_subset --output-dir images/mit_indoor_aug --manifest sam_output/aug_manifest.json --augs-per-image 1

Generated 120 augmented images in images/mit_indoor_aug
Manifest: sam_output/aug_manifest.json


## 6. Run SAM on augmented images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_aug --output sam_output/augmented --checkpoint sam_vit_b_01ec64.pth --model vit_b

Images: 100% 120/120 [50:49<00:00, 25.41s/it]
Done. Processed 120 images. Output: sam_output/augmented


## 7. Evaluate consistency, analyze, plot

In [ ]:
!python evaluate_consistency.py --original-output sam_output/original --augmented-output sam_output/augmented --manifest sam_output/aug_manifest.json --results sam_output/consistency_results.json

Results saved to sam_output/consistency_results.json

SAM MASK CONSISTENCY (original vs augmented)
  Pairs evaluated:     120
  Mean best IoU:      0.5194
  Median best IoU:    0.5095
  Survival @ 0.5:     56.50% (masks with a match >= 0.5 IoU)
  Survival @ 0.7:     51.04%

Per image (original_stem / augmentation):
  bedroom__375 | hflip    | mean_iou=0.756 survival_0.5=79.59% n_orig=49 n_aug=54
  bedroom__IMG_0028 | rotation | mean_iou=0.726 survival_0.5=76.39% n_orig=72 n_aug=75
  bedroom__IMG_0994 | resize   | mean_iou=0.750 survival_0.5=79.59% n_orig=49 n_aug=43
  bedroom__IMG_1080 | blur     | mean_iou=0.709 survival_0.5=74.59% n_orig=122 n_aug=96
  bedroom__IMG_1098 | hflip    | mean_iou=0.848 survival_0.5=87.76% n_orig=49 n_aug=44
  bedroom__Madison-room | rotation | mean_iou=0.330 survival_0.5=35.51% n_orig=138 n_aug=62
  bedroom__N190002 | resize   | mean_iou=0.535 survival_0.5=59.26% n_orig=108 n_aug=84
  bedroom__N190091 | blur     | mean_iou=0.493 survival_0.5=56.25% n_orig

In [ ]:
!python analyze_consistency.py --results sam_output/consistency_results.json --original-metadata sam_output/original/metadata --output sam_output/analysis.json

Analysis saved to sam_output/analysis.json


In [ ]:
!python plot_consistency.py --analysis sam_output/analysis.json --output-dir sam_output/plots

Plots saved to sam_output/plots


## 8. Zip results and download (optional)

In [ ]:
os.chdir(PROJECT_DIR)
!zip -r sam_pipeline_results.zip sam_output/ images/mit_indoor_subset/ images/mit_indoor_aug/ -x "*.pyc" 2>/dev/null
from google.colab import files
zip_path = os.path.join(PROJECT_DIR, "sam_pipeline_results.zip")
if os.path.isfile(zip_path):
    files.download(zip_path)
    print("Download started: sam_pipeline_results.zip")
else:
    print("Zip not found; download sam_output/ and images/ from the file browser.")

Streaming output truncated to the last 5000 lines.
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0013.png (deflated 22%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0019.png (deflated 24%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0007.png (deflated 60%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0024.png (deflated 24%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0020.png (deflated 16%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0030.png (deflated 35%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0011.png (deflated 26%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0027.png (deflated 26%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0014.png (deflated 27%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0017.png (deflated 26%)
  adding: sam_output/original/masks/livingroom__cdMC1284/mask_0021.png (deflated 19%)
  a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started: sam_pipeline_results.zip
